# Sentiment Pipeline Demo

This notebook demonstrates the sentiment embedding pipeline from `sentiment.embeddings`.

**What it does:** `Article` dict &rarr; BART-CNN summary &rarr; FinBERT &rarr; binary sentiment label + 768-dim embedding.

Run the cells in order. Model loading (cell 1) takes ~30 seconds on first run.

In [1]:
from sentiment.embeddings import SentimentPipeline, aggregate_daily

pipe = SentimentPipeline()
print("Models loaded.")

c:\_Finance\TFA\quant-sentiment-score\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13365.70it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded.


In [ ]:
from datetime import date

# Process a single Article through the full pipeline
article = {
    "id": "demo-1",
    "url": "https://example.com/apple-revenue",
    "title": "Apple reports record quarterly revenue",
    "text": (
        "Apple Inc. reported record quarterly revenue of $124 billion for the fiscal "
        "first quarter of 2025, up 11% year over year. The company saw strong demand "
        "across its product lineup, with iPhone revenue reaching an all-time high of "
        "$71 billion. Services revenue also set a new record at $26 billion."
    ),
    "publish_date": date(2025, 1, 15),
    "source_name": "example.com",
    "language": "en",
}

result = pipe.process_article(article)

print(f"Summary:   {result['summary'][:120]}...")
print(f"Label:     {result['label']}  (1=positive, 0=negative/neutral)")
print(f"Embedding: shape={result['embedding'].shape}, dtype={result['embedding'].dtype}")

In [ ]:
# Use process_ticker_articles for the full end-to-end flow
ticker_articles = {
    "AAPL": [
        {"id": "a1", "url": "https://example.com/1", "title": "Apple reports record quarterly revenue",
         "text": "Strong iPhone and services demand drove results.",
         "publish_date": date(2025, 1, 15), "source_name": "example.com", "language": "en"},
        {"id": "a2", "url": "https://example.com/2", "title": "Apple faces regulatory scrutiny over App Store",
         "text": "EU regulators opened an investigation into anti-competitive practices.",
         "publish_date": date(2025, 1, 15), "source_name": "example.com", "language": "en"},
        {"id": "a3", "url": "https://example.com/3", "title": "Apple expands AI features across product lineup",
         "text": "New AI capabilities announced for iPhone, iPad, and Mac.",
         "publish_date": date(2025, 1, 15), "source_name": "example.com", "language": "en"},
        {"id": "a4", "url": "https://example.com/4", "title": "Apple stock dips on profit-taking after earnings",
         "text": "Shares fell 3% as investors took profits.",
         "publish_date": date(2025, 1, 16), "source_name": "example.com", "language": "en"},
    ],
    "MSFT": [
        {"id": "m1", "url": "https://example.com/5", "title": "Microsoft cloud revenue surges past expectations",
         "text": "Azure cloud revenue grew 29%, beating analyst estimates.",
         "publish_date": date(2025, 1, 15), "source_name": "example.com", "language": "en"},
        {"id": "m2", "url": "https://example.com/6", "title": "Microsoft announces layoffs in gaming division",
         "text": "1,900 employees affected following the Activision acquisition.",
         "publish_date": date(2025, 1, 15), "source_name": "example.com", "language": "en"},
    ],
}

daily = pipe.process_ticker_articles(ticker_articles)

print("Daily aggregated results:")
print(daily[["ticker", "date", "sentiment_score", "n_articles"]].to_string(index=False))

In [ ]:
import numpy as np

for _, row in daily.iterrows():
    emb = row["embedding"]
    print(f"  {row['ticker']} {row['date']}: emb_shape={emb.shape}, finite={np.all(np.isfinite(emb))}")

In [ ]:
daily